# 准备数据

In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 减少TF的警告信息

def mnist_dataset():
    # 直接从指定路径加载npz文件
    data = np.load('E:\\My struggle\\SVM 手写数字\\mnist.npz')
    x, y = data['x_train'], data['y_train']
    x_test, y_test = data['x_test'], data['y_test']
    
    # 归一化
    x = x.astype(np.float32) / 255.0
    x_test = x_test.astype(np.float32) / 255.0
    
    return (x, y), (x_test, y_test)

# 建立模型

In [10]:
class myModel:
    def __init__(self):
        # 声明模型参数
        # 输入层: 784 (28x28)
        # 隐藏层: 256 
        # 输出层: 10 (数字0-9)
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        # 实现前向传播
        # 将输入reshape为[batch_size, 784]
        x = tf.reshape(x, [-1, 784])
        # 隐藏层，使用ReLU激活函数
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 输出层，返回logits
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()
optimizer = optimizers.Adam(learning_rate=0.001)

# 计算LOSS

In [11]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        # 前向传播
        logits = model(x)
        # 计算损失
        loss = compute_loss(logits, y)
        
    # 计算梯度
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    # 应用梯度
    optimizer.apply_gradients(zip(grads, trainable_vars))
    
    # 计算准确率
    accuracy = compute_accuracy(logits, y)
    
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy



# 实际训练

In [12]:
train_data, test_data = mnist_dataset()

# 设置批次大小和数据加载器
batch_size = 128
train_dataset = tf.data.Dataset.from_tensor_slices((train_data[0], train_data[1]))
train_dataset = train_dataset.shuffle(10000).batch(batch_size)

# 训练循环
for epoch in range(50):
    # 初始化统计变量
    total_loss = 0.0
    total_accuracy = 0.0
    steps = 0
    
    # 批次训练
    for x, y in train_dataset:
        # 确保数据类型正确
        x = tf.cast(x, tf.float32)
        y = tf.cast(y, tf.int64)
        
        # 训练一个批次
        loss, accuracy = train_one_step(model, optimizer, x, y)
        
        # 累计统计信息
        total_loss += loss
        total_accuracy += accuracy
        steps += 1
    
    # 计算平均值
    avg_loss = total_loss / steps
    avg_accuracy = total_accuracy / steps
    print(f'epoch {epoch}: loss {avg_loss:.4f}; accuracy {avg_accuracy:.4f}')

# 测试
test_dataset = tf.data.Dataset.from_tensor_slices((test_data[0], test_data[1])).batch(batch_size)
test_loss = 0.0
test_accuracy = 0.0
test_steps = 0

for x, y in test_dataset:
    x = tf.cast(x, tf.float32)
    y = tf.cast(y, tf.int64)
    batch_loss, batch_accuracy = test(model, x, y)
    test_loss += batch_loss
    test_accuracy += batch_accuracy
    test_steps += 1

test_loss /= test_steps
test_accuracy /= test_steps
print(f'test loss {test_loss:.4f}; accuracy {test_accuracy:.4f}')

epoch 0: loss 0.3158; accuracy 0.9092
epoch 1: loss 0.1317; accuracy 0.9630
epoch 2: loss 0.0891; accuracy 0.9744
epoch 3: loss 0.0662; accuracy 0.9810
epoch 4: loss 0.0505; accuracy 0.9860
epoch 5: loss 0.0394; accuracy 0.9888
epoch 6: loss 0.0322; accuracy 0.9908
epoch 7: loss 0.0252; accuracy 0.9932
epoch 8: loss 0.0196; accuracy 0.9951
epoch 9: loss 0.0152; accuracy 0.9965
epoch 10: loss 0.0127; accuracy 0.9973
epoch 11: loss 0.0101; accuracy 0.9980
epoch 12: loss 0.0083; accuracy 0.9984
epoch 13: loss 0.0071; accuracy 0.9986
epoch 14: loss 0.0066; accuracy 0.9987
epoch 15: loss 0.0076; accuracy 0.9982
epoch 16: loss 0.0040; accuracy 0.9994
epoch 17: loss 0.0019; accuracy 0.9999
epoch 18: loss 0.0035; accuracy 0.9993
epoch 19: loss 0.0051; accuracy 0.9989
epoch 20: loss 0.0053; accuracy 0.9988
epoch 21: loss 0.0050; accuracy 0.9984
epoch 22: loss 0.0021; accuracy 0.9996
epoch 23: loss 0.0006; accuracy 1.0000
epoch 24: loss 0.0004; accuracy 1.0000
epoch 25: loss 0.0004; accuracy 1.0